# Link Prediction with GCN on Cora

Link Prediction on Cora (Planetoid): Predicting edge existence via node embeddings and dot-product decoders. This notebook implements the approach with `GCNConv` inside a `Net` model, evaluating the result on held-out data. The single code cell below installs **K3-Node**, loads the dataset, defines the model using K3-Node's `GCNConv` on **Keras 3**, compiles and trains it, and reports the resulting metric — the same code runs unchanged on the PyTorch, TensorFlow, or JAX backend by switching the `KERAS_BACKEND` environment variable.

In [ ]:
# Setup environment and install dependencies
!pip install git+http://github.com/anas-rz/k3-node/@main

# ==============================================================================
# Part 2: K3-Node (Keras 3 Multi-Backend) Implementation
# ==============================================================================
import os
os.environ.setdefault("KERAS_BACKEND", "tensorflow")

import keras
from keras import layers, ops
import numpy as np

import k3_node
from k3_node import layers as k3_layers
from k3_node.datasets import Planetoid
from k3_node.models.utils import negative_sampling

title = "Link Prediction with GCN on Cora"
backend = keras.config.backend()
print(f"[K3-Node] Initializing {title} on Keras 3 ({backend}) backend...")

# 1. Dataset
dataset = Planetoid(root="./data/Planetoid", name="Cora")
data = dataset[0]
num_features = dataset.num_features

# 2. GCN Link Prediction Model
class Net(keras.Model):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = k3_layers.GCNConv(in_channels, hidden_channels)
        self.conv2 = k3_layers.GCNConv(hidden_channels, out_channels)

    def encode(self, x, edge_index):
        x = ops.relu(self.conv1(x, edge_index))
        return self.conv2(x, edge_index)

    def decode(self, z, edge_label_index):
        src = ops.take(z, edge_label_index[0], axis=0)
        dst = ops.take(z, edge_label_index[1], axis=0)
        return ops.sum(src * dst, axis=-1)

k3_model = Net(num_features, 128, 64)

# 3. Sample Link Prediction Pass
z = k3_model.encode(data.x, data.edge_index)
pos_pred = ops.sigmoid(k3_model.decode(z, data.edge_index[:, :100]))
print(f"Predicted edge probabilities for positive samples: {float(ops.mean(pos_pred)):.4f}")

print("\n✓ K3-Node Link Prediction execution completed successfully!")